This code generates the combined csv that are produced by the pruning and benchmarking scripts. A few tinkers need to be made to the CSVs, like removing the unnecessary rows and merging the values for energy and performancy on to the same row, but once the individual rows are representative of a single model, this code will combine them.

In [ ]:
import pandas as pd
import os

# Define the base directory where your CNN folder is located
base_dir = '/Users/arihangupta/Downloads/pruning_project_data/CNN'

# Define the datasets to process
datasets = ['bloodmnist', 'dermamnist', 'pathmnist']

for dataset in datasets:
    print(f"\nProcessing {dataset}...")
    
    # Define file paths with absolute paths
    metrics_file = os.path.join(base_dir, f'CNN_pruned_models/{dataset}/{dataset}_combined_pruning_kd_metrics_with_energy.csv')
    timing_file = os.path.join(base_dir, f'timing_exps/{dataset}_results.csv')
    output_file = os.path.join(base_dir, f'{dataset}_combined_results.csv')
    
    # Check if files exist
    if not os.path.exists(metrics_file):
        print(f"  Warning: {metrics_file} not found, skipping...")
        continue
    if not os.path.exists(timing_file):
        print(f"  Warning: {timing_file} not found, skipping...")
        continue
    
    # Read the CSV files
    df_metrics = pd.read_csv(metrics_file)
    df_timing = pd.read_csv(timing_file)
    
    # Remove any unnamed columns from both dataframes
    df_metrics = df_metrics.loc[:, ~df_metrics.columns.str.contains('^Unnamed')]
    df_timing = df_timing.loc[:, ~df_timing.columns.str.contains('^Unnamed')]
    
    print(f"  Metrics file shape: {df_metrics.shape}")
    print(f"  Timing file shape: {df_timing.shape}")
    
    # Perform inner join on Variant (from metrics) and pruning_method (from timing)
    df_combined = pd.merge(
        df_metrics, 
        df_timing, 
        left_on='Variant', 
        right_on='pruning_method', 
        how='inner'
    )
    
    print(f"  Combined file shape: {df_combined.shape}")
    print(f"  Matched {len(df_combined)} rows")
    
    # Save the combined file
    df_combined.to_csv(output_file, index=False)
    print(f"  Saved to: {output_file}")

print("\nAll datasets processed!")

This code works to average all of the results (now in one file per dataset) and also store the the SD. This merged result is then used for visualization.

In [ ]:
import pandas as pd
import os
import numpy as np

# Define the base directory
base_dir = '/Users/arihangupta/Downloads/pruning_project_data/CNN'
output_dir = os.path.join(base_dir, 'merged_results')

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the datasets to process
datasets = ['bloodmnist', 'dermamnist', 'pathmnist']

# Define columns that should be averaged (numerical columns that make sense to average)
columns_to_average = [
    'Acc', 'AUC', 'Loss',
    'InferenceTime_per_batch_s', 'PeakRAM_MB',
    'RetrainEnergy_kWh', 'RetrainEmissions_kg',
    'images_processed', 'elapsed_s', 'throughput_imgs_per_s',
    'auc', 'median_batch_ms', 'p50_ms', 'p90_ms',
    'peak_gpu_mem_MB', 'avg_power_W',
    'energy_kWh_total', 'energy_kWh_per_batch', 'energy_kWh_per_image',
    'emissions_kg_total', 'cpu_power_w', 'gpu_power_w', 'ram_power_w'
]

# Define grouping columns
grouping_columns = [
    'model_name', 'pruning_method', 'sparsity', 
    'stored_precision', 'batch_size', 'runtime_precision'
]

for dataset in datasets:
    print(f"\nProcessing {dataset}...")
    
    # Read the combined results file
    input_file = os.path.join(base_dir, f'{dataset}_combined_results.csv')
    
    if not os.path.exists(input_file):
        print(f"  Warning: {input_file} not found, skipping...")
        continue
    
    df = pd.read_csv(input_file)
    print(f"  Original shape: {df.shape}")
    
    # Filter out rows where stored_precision != runtime_precision
    df_filtered = df[df['stored_precision'] == df['runtime_precision']].copy()
    print(f"  After filtering mismatched precisions: {df_filtered.shape}")
    
    # Group by the specified columns and aggregate
    agg_dict = {}
    
    # For columns to average, use both mean and std
    for col in columns_to_average:
        if col in df_filtered.columns:
            agg_dict[col] = ['mean', 'std']
    
    # For other columns, take the first value (they should be the same within groups)
    for col in df_filtered.columns:
        if col not in grouping_columns and col not in columns_to_average and col not in ['run_id', 'rep']:
            agg_dict[col] = 'first'
    
    # Perform the aggregation
    df_averaged = df_filtered.groupby(grouping_columns, as_index=False).agg(agg_dict)
    
    # Flatten the multi-level column names
    new_columns = []
    for col in df_averaged.columns:
        if isinstance(col, tuple):
            if col[1] == 'mean':
                new_columns.append(col[0])
            elif col[1] == 'std':
                new_columns.append(f"{col[0]}_sd")
            else:
                new_columns.append(col[0])
        else:
            new_columns.append(col)
    
    df_averaged.columns = new_columns
    
    # Reorder columns so that each averaged column is followed by its _sd column
    final_columns = []
    processed_cols = set()
    
    for col in df_averaged.columns:
        if col not in processed_cols:
            final_columns.append(col)
            processed_cols.add(col)
            # Check if there's a corresponding _sd column
            sd_col = f"{col}_sd"
            if sd_col in df_averaged.columns and sd_col not in processed_cols:
                final_columns.append(sd_col)
                processed_cols.add(sd_col)
    
    df_averaged = df_averaged[final_columns]
    
    print(f"  After averaging runs: {df_averaged.shape}")
    print(f"  Averaged {len(df_filtered) - len(df_averaged)} rows into {len(df_averaged)} groups")
    
    # Save the result
    output_file = os.path.join(output_dir, f'{dataset}_averaged_results.csv')
    df_averaged.to_csv(output_file, index=False)
    print(f"  Saved to: {output_file}")

print("\nAll datasets processed!")
print(f"Results saved to: {output_dir}")

In [ ]:
"""
Save CNN Model Performance Data to LaTeX Format and Generate Radar Plots

This script processes CSV data, generates radar plots, and saves raw metrics 
to LaTeX table format.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import pi
import os


def create_method_label(row):
    """Create display labels for each method variant."""
    method = row['pruning_method']
    precision = row['runtime_precision']
    sparsity = row['sparsity']
    
    if method == 'baseline' and precision == 'fp32':
        return 'Baseline (FP32)', 'baseline', 0
    elif method == 'quantization' and precision == 'amp':
        return 'Quantized (AMP)', 'quantization', 1
    elif method == 'hybrid_pruning' and precision == 'fp32':
        return f'Hybrid Pruning {sparsity} (FP32)', 'hybrid_pruning', 2
    elif method == 'hybrid_pruning_fp16' and precision == 'amp':
        return f'Hybrid Pruning {sparsity} (AMP)', 'hybrid_pruning_fp16', 3
    elif method == 'regional_pruning' and precision == 'fp32':
        return f'Regional Pruning {sparsity} (FP32)', 'regional_pruning', 4
    elif method == 'regional_pruning_fp16' and precision == 'amp':
        return f'Regional Pruning {sparsity} (AMP)', 'regional_pruning_fp16', 5
    elif method == 'slim_kd' and precision == 'fp32':
        return f'Slim KD {sparsity} (FP32)', 'slim_kd', 6
    elif method == 'slim_kd_fp16' and precision == 'amp':
        return f'Slim KD {sparsity} (AMP)', 'slim_kd_fp16', 7
    else:
        return f'{method} {precision}', 'other', 99


def get_method_color(label):
    """Get color for each method label"""
    color_map = {
        'Baseline (FP32)': '#FF0000',
        'Quantized (AMP)': '#228B22',
        'Hybrid Pruning 50% (FP32)': '#0000CD',
        'Hybrid Pruning 50% (AMP)': '#4169E1',
        'Regional Pruning 50% (FP32)': '#6A0DAD',
        'Regional Pruning 50% (AMP)': '#9370DB',
        'Slim KD 50% (FP32)': '#FF8C00',
        'Slim KD 50% (AMP)': '#FFA500',
    }
    return color_map.get(label, '#333333')


def save_data_to_latex(csv_path, output_dir='/Users/arihangupta/Downloads/pruning_project_data/CNN/Visuals', dataset_name='bloodmnist'):
    """Extract and save raw data to LaTeX format text files for manual plotting."""
    
    os.makedirs(output_dir, exist_ok=True)
    
    df = pd.read_csv(csv_path)
    print(f"Loaded CSV with {len(df)} rows")
    
    if 'batch_size' not in df.columns:
        raise KeyError("'batch_size' column not found in CSV")
    
    df[['method_label', 'method_group', 'sort_order']] = df.apply(
        lambda row: pd.Series(create_method_label(row)), axis=1
    )
    
    if df['Acc'].max() > 1.5:
        df['Acc'] = df['Acc'] / 100.0
    if df['AUC'].max() > 1.5:
        df['AUC'] = df['AUC'] / 100.0
    
    df['energy_kWh_per_image'] = df['energy_kWh_per_image'].replace(0, np.nan)
    df['peak_gpu_mem_MB'] = df['peak_gpu_mem_MB'].replace(0, np.nan)
    df['ModelSizeMB'] = df['ModelSizeMB'].replace(0, np.nan)
    
    batch_sizes = sorted(df['batch_size'].unique())
    
    for batch_size in batch_sizes:
        print(f"\n{'='*60}")
        print(f"Processing Batch Size {batch_size}")
        print(f"{'='*60}")
        
        batch_df = df[df['batch_size'] == batch_size].copy()
        
        if batch_df.empty:
            continue
        
        baseline_candidates = batch_df[batch_df['method_label'] == 'Baseline (FP32)']
        
        if baseline_candidates.empty:
            continue
        
        baseline = baseline_candidates.sort_values('Acc', ascending=False).iloc[0]
        
        # Calculate ratios relative to baseline
        batch_df['throughput_ratio'] = batch_df['throughput_imgs_per_s'] / baseline['throughput_imgs_per_s']
        batch_df['energy_ratio'] = baseline['energy_kWh_per_image'] / batch_df['energy_kWh_per_image']
        batch_df['ram_ratio'] = baseline['peak_gpu_mem_MB'] / batch_df['peak_gpu_mem_MB']
        batch_df['modelsize_ratio'] = baseline['ModelSizeMB'] / batch_df['ModelSizeMB']
        
        max_acc = 1.0
        max_auc = 1.0
        max_throughput_ratio = batch_df['throughput_ratio'].max()
        max_energy_ratio = batch_df['energy_ratio'].max()
        max_ram_ratio = batch_df['ram_ratio'].max()
        max_modelsize_ratio = batch_df['modelsize_ratio'].max()
        
        models_to_plot = []
        
        for _, row in batch_df.iterrows():
            label = row['method_label']
            
            acc = row['Acc']
            auc = row['AUC']
            throughput = row['throughput_imgs_per_s']
            energy = row['energy_kWh_per_image']
            ram = row['peak_gpu_mem_MB']
            modelsize = row['ModelSizeMB']
            
            if pd.isna(acc) or pd.isna(auc) or pd.isna(throughput) or pd.isna(energy) or pd.isna(ram) or pd.isna(modelsize):
                continue
            
            normalized_values = []
            normalized_values.append(acc / max_acc)
            normalized_values.append(auc / max_auc)
            
            throughput_ratio = throughput / baseline['throughput_imgs_per_s']
            normalized_values.append(min(throughput_ratio / max_throughput_ratio, 1.0))
            
            energy_ratio = baseline['energy_kWh_per_image'] / energy
            normalized_values.append(min(energy_ratio / max_energy_ratio, 1.0))
            
            ram_ratio = baseline['peak_gpu_mem_MB'] / ram
            normalized_values.append(min(ram_ratio / max_ram_ratio, 1.0))
            
            modelsize_ratio = baseline['ModelSizeMB'] / modelsize
            normalized_values.append(min(modelsize_ratio / max_modelsize_ratio, 1.0))
            
            models_to_plot.append({
                'label': label,
                'values': normalized_values,
                'raw_values': {
                    'Acc': acc,
                    'AUC': auc,
                    'Throughput': throughput,
                    'Energy': energy,
                    'RAM': ram,
                    'ModelSize': modelsize
                },
                'sort_order': row['sort_order']
            })
        
        models_to_plot.sort(key=lambda x: x['sort_order'])
        
        # Create LaTeX file with raw data
        latex_filename = os.path.join(output_dir, f'{dataset_name}_batch{batch_size}_data.tex')
        
        with open(latex_filename, 'w') as f:
            # Write header
            f.write("% Raw data for radar chart\n")
            f.write(f"% Dataset: {dataset_name}, Batch Size: {batch_size}\n\n")
            
            # Write baseline values
            f.write("% Baseline Values\n")
            f.write("\\begin{table}[h]\n")
            f.write("\\centering\n")
            f.write("\\begin{tabular}{ll}\n")
            f.write("\\toprule\n")
            f.write("Metric & Value \\\\\n")
            f.write("\\midrule\n")
            f.write(f"Accuracy & {baseline['Acc']:.4f} \\\\\n")
            f.write(f"AUC & {baseline['AUC']:.6f} \\\\\n")
            f.write(f"Throughput (img/s) & {baseline['throughput_imgs_per_s']:.2f} \\\\\n")
            f.write(f"Energy (kWh/img) & {baseline['energy_kWh_per_image']:.2e} \\\\\n")
            f.write(f"Peak GPU RAM (MB) & {baseline['peak_gpu_mem_MB']:.1f} \\\\\n")
            f.write(f"Model Size (MB) & {baseline['ModelSizeMB']:.1f} \\\\\n")
            f.write("\\bottomrule\n")
            f.write("\\end{tabular}\n")
            f.write("\\caption{Baseline Values}\n")
            f.write("\\end{table}\n\n")
            
            # Write raw metrics table
            f.write("% Raw Metrics for All Models\n")
            f.write("\\begin{table}[h]\n")
            f.write("\\centering\n")
            f.write("\\small\n")
            f.write("\\begin{tabular}{lcccccc}\n")
            f.write("\\toprule\n")
            f.write("Method & Acc & AUC & Throughput & Energy & GPU RAM & Size \\\\\n")
            f.write(" & & & (img/s) & (kWh/img) & (MB) & (MB) \\\\\n")
            f.write("\\midrule\n")
            
            for model in models_to_plot:
                raw = model['raw_values']
                # Escape special characters for LaTeX
                method_name = model['label'].replace('_', '\\_').replace('%', '\\%').replace('&', '\\&')
                f.write(f"{method_name} & {raw['Acc']:.4f} & {raw['AUC']:.6f} & "
                       f"{raw['Throughput']:.2f} & {raw['Energy']:.2e} & "
                       f"{raw['RAM']:.1f} & {raw['ModelSize']:.1f} \\\\\n")
            
            f.write("\\bottomrule\n")
            f.write("\\end{tabular}\n")
            f.write("\\caption{Raw Performance Metrics}\n")
            f.write("\\end{table}\n\n")
            
            # Write normalized values table
            f.write("% Normalized Values for Radar Plot (0-1 scale)\n")
            f.write("\\begin{table}[h]\n")
            f.write("\\centering\n")
            f.write("\\small\n")
            f.write("\\begin{tabular}{lcccccc}\n")
            f.write("\\toprule\n")
            f.write("Method & Acc & AUC & Throughput & Energy & RAM & Size \\\\\n")
            f.write(" & (norm) & (norm) & (ratio) & (ratio) & (ratio) & (ratio) \\\\\n")
            f.write("\\midrule\n")
            
            for model in models_to_plot:
                method_name = model['label'].replace('_', '\\_').replace('%', '\\%').replace('&', '\\&')
                vals = model['values']
                f.write(f"{method_name} & {vals[0]:.4f} & {vals[1]:.4f} & "
                       f"{vals[2]:.4f} & {vals[3]:.4f} & {vals[4]:.4f} & {vals[5]:.4f} \\\\\n")
            
            f.write("\\bottomrule\n")
            f.write("\\end{tabular}\n")
            f.write("\\caption{Normalized Values for Radar Chart (Baseline = 1.0)}\n")
            f.write("\\end{table}\n\n")
            
            # Write scaling factors
            f.write("% Scaling Factors Used\n")
            f.write("\\begin{table}[h]\n")
            f.write("\\centering\n")
            f.write("\\begin{tabular}{ll}\n")
            f.write("\\toprule\n")
            f.write("Metric & Max Ratio \\\\\n")
            f.write("\\midrule\n")
            f.write(f"Accuracy & {max_acc:.4f} (absolute) \\\\\n")
            f.write(f"AUC & {max_auc:.6f} (absolute) \\\\\n")
            f.write(f"Throughput & {max_throughput_ratio:.2f}x baseline \\\\\n")
            f.write(f"Energy & {max_energy_ratio:.2f}x (baseline/current) \\\\\n")
            f.write(f"RAM & {max_ram_ratio:.2f}x (baseline/current) \\\\\n")
            f.write(f"Model Size & {max_modelsize_ratio:.2f}x (baseline/current) \\\\\n")
            f.write("\\bottomrule\n")
            f.write("\\end{tabular}\n")
            f.write("\\caption{Scaling Factors}\n")
            f.write("\\end{table}\n\n")
            
            # Write Python/JSON-like data for easy parsing
            f.write("% Python-readable format for plotting\n")
            f.write("% models = [\n")
            for model in models_to_plot:
                f.write(f"%   {{\n")
                f.write(f"%     'label': '{model['label']}',\n")
                f.write(f"%     'normalized': {model['values']},\n")
                raw = model['raw_values']
                f.write(f"%     'raw': {{'Acc': {raw['Acc']:.4f}, 'AUC': {raw['AUC']:.6f}, ")
                f.write(f"'Throughput': {raw['Throughput']:.2f}, 'Energy': {raw['Energy']:.2e}, ")
                f.write(f"'RAM': {raw['RAM']:.1f}, 'ModelSize': {raw['ModelSize']:.1f}}}\n")
                f.write(f"%   }},\n")
            f.write("% ]\n")
        
        print(f"LaTeX data file saved to: {latex_filename}")
        
        # Generate radar plot
        num_vars = 6  # 6 metrics
        angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
        angles += angles[:1]
        
        fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(projection='polar'))
        
        # Plot each model
        for idx, model_data in enumerate(models_to_plot):
            values = model_data['values']
            values += values[:1]
            
            linewidth = 3 if 'Baseline' in model_data['label'] else 2
            alpha_line = 1.0 if 'Baseline' in model_data['label'] else 0.8
            alpha_fill = 0.25 if 'Baseline' in model_data['label'] else 0.15
            
            color = get_method_color(model_data['label'])
            
            ax.plot(angles, values, 'o-', linewidth=linewidth, label=model_data['label'], 
                    color=color, markersize=8, alpha=alpha_line)
            ax.fill(angles, values, alpha=alpha_fill, color=color)
        
        # Set axis labels
        ax.set_xticks(angles[:-1])
        metric_labels = ['Accuracy', 'AUC', 'Throughput\n(imgs/s)', 'Energy\n(lower=better)', 
                        'Peak GPU RAM\n(lower=better)', 'Model Size\n(lower=better)']
        
        axis_labels = []
        metric_keys = ['Acc', 'AUC', 'throughput_imgs_per_s', 'energy_kWh_per_image', 'peak_gpu_mem_MB', 'ModelSizeMB']
        for i, (metric_label, metric_key) in enumerate(zip(metric_labels, metric_keys)):
            baseline_val = baseline[metric_key]
            if metric_key == 'Acc':
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.3f})")
            elif metric_key == 'AUC':
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.4f})")
            elif metric_key == 'throughput_imgs_per_s':
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.1f} img/s)")
            elif metric_key == 'energy_kWh_per_image':
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.2e} kWh)")
            elif metric_key == 'peak_gpu_mem_MB':
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.1f} MB)")
            else:
                axis_labels.append(f"{metric_label}\n(baseline: {baseline_val:.1f} MB)")
        
        ax.set_xticklabels(axis_labels, size=11)
        
        # Set y-axis
        ax.set_ylim(0, 1.0)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=10)
        ax.grid(True, linestyle='--', alpha=0.7)
        
        # Add title
        dataset_display = dataset_name.replace('mnist', ' MNIST').title()
        plt.title(f'CNN Optimization Methods - {dataset_display} (Batch Size {batch_size})\nAll Methods Normalized to Baseline = 1.0',
                  size=16, weight='bold', pad=20)
        
        # Add legend
        plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), fontsize=11)
        
        # Save figure
        plot_filename = os.path.join(output_dir, f'cnn_radar_{dataset_name}_batch{batch_size}.png')
        plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
        print(f"Radar plot saved to: {plot_filename}")
        
        plt.show()
        plt.close()
        
        # Print summary to console
        print(f"\nPerformance Summary (Batch Size {batch_size})")
        print("="*140)
        print(f"{'Model':<35} {'Accuracy':<12} {'AUC':<12} {'Throughput':<15} {'Energy/img':<15} {'GPU RAM (MB)':<15} {'Size (MB)':<12}")
        print("-"*140)
        
        for model in models_to_plot:
            raw = model['raw_values']
            print(f"{model['label']:<35} {raw['Acc']:<12.4f} {raw['AUC']:<12.6f} "
                  f"{raw['Throughput']:<15.2f} {raw['Energy']:<15.2e} {raw['RAM']:<15.1f} {raw['ModelSize']:<12.1f}")
        print("="*140)


if __name__ == "__main__":
    base_dir = "/Users/arihangupta/Downloads/pruning_project_data/CNN"
    output_dir = os.path.join(base_dir, "Visuals")
    
    datasets = {
        'bloodmnist': os.path.join(base_dir, "merged_results/bloodmnist_averaged_results.csv"),
        'dermamnist': os.path.join(base_dir, "merged_results/dermamnist_averaged_results.csv"),
        'pathmnist': os.path.join(base_dir, "merged_results/pathmnist_averaged_results.csv")
    }
    
    for dataset_name, csv_path in datasets.items():
        print(f"\n{'#'*80}")
        print(f"# Processing {dataset_name.upper()}")
        print(f"{'#'*80}")
        
        try:
            save_data_to_latex(csv_path, output_dir=output_dir, dataset_name=dataset_name)
        except Exception as e:
            print(f"Error processing {dataset_name}: {e}")
            continue